# VRSBench Image Captioning — Moondream2 + LoRA + Length-Penalty Loss (8hr, best-checkpoint run)

**Changes vs. the 9hr baseline run:**
- Wall-clock training cap lowered from 9hr → **8hr** (`TARGET_TRAIN_MINUTES = 480`).
- Length-penalty weight lowered from `0.05` → **`0.02`**: the earlier value was pulling hard enough
  to fight the content loss and clip detail out of captions; the goal is still to land near the
  53-word VRSBench median, but as a soft nudge, not a dominant loss term.
- Training now runs a **periodic lightweight eval** (every 30 min, 12-sample subset) against the
  official val split, and checkpoints the LoRA adapter to `BEST_OUTPUT_DIR` whenever BERT-BLEU4
  improves — so the notebook keeps the single best-scoring checkpoint seen during the whole 8hr
  run, not just whatever state the model happens to be in when the clock runs out.
- The final cell runs the full 50-sample official eval on **both** the best checkpoint and the
  last checkpoint and reports both, so you can see whether early stopping actually helped.
- The BERT-BLEU4 metric itself is **unchanged** from the baseline run and is the exact formula
  used in both reference reports (GeoNLI and Team 67 / GeoSpatial NLI):
  `BERT-BLEU4 = LP · exp( (1/4) · Σ_{n=1..4} log P_n )`, `LP = exp(-α·|L_C - L_R|/L_R)`, `α = 0.5`,
  with `P_n` the max-cosine-similarity BERT-embedding n-gram recall. This lets the score reported
  at the end be read directly against the reports' tables (e.g. GeoNLI's best SFT run: 0.6841;
  Team 67's unified end-to-end run: 0.7993).

Moondream2 does **not** use a standard chat-template + `Trainer` pipeline like SmolVLM — it's a
custom `trust_remote_code=True` architecture (a Phi-based text decoder + a separate vision encoder,
loaded via `AutoModelForCausalLM`) that builds its own `[BOS] + image_embeds + question_tokens +
answer_tokens` input sequence and doesn't expose a HF-`Trainer`-compatible `forward(labels=...)`
in the way SmolVLM's `AutoModelForImageTextToText` does. So Section 9 uses a manual training loop
instead of `Trainer`, built on the pattern from the **official moondream finetuning notebook**
(`m87-labs/moondream`, PR #227/#250, `finetune/finetune_text.py` — LoRA/full finetune of
`text_model` only, vision encoder frozen) and cross-checked against public moondream2 finetunes on
the Hub (e.g. `zesquirrelnator/moondream2-finetuneV2`, `gokaygokay/moondream-prompt`) for the
`encode_image` / `answer_question` / prompt-format conventions.

## 1. Setup — installs, imports, data + model download (everything up front)

In [1]:
!pip install -q --no-deps "transformers==4.43.4"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 59.7 MB/s eta 0:00:00


In [2]:
!pip install -q accelerate peft bitsandbytes einops timm num2words
!pip install -q sentence-transformers nltk rouge-score pycocoevalcap


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 17.8 MB/s eta 0:00:00


In [3]:
import numpy as np
import transformers
import torch

print("NumPy:", np.__version__)
print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

NumPy: 2.0.2
Transformers: 4.43.4
PyTorch: 2.10.0+cu128
CUDA: True


In [4]:
import os
import json
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.bfloat16 if (DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16
print("Device:", DEVICE, "| dtype:", DTYPE)

RUN_START = time.time()
def elapsed_min():
    return (time.time() - RUN_START) / 60

Device: cuda | dtype: torch.bfloat16


In [5]:
# --- Dataset paths (mounted Kaggle input, no download needed) ---
DATA_DIR = "/kaggle/input/datasets/ayaanmustafa/vrs-bench/vrsbench_data"
ANNOT_FILE = os.path.join(DATA_DIR, "VRSBench_train_caption.json")
IMG_DIR = os.path.join(DATA_DIR, "Images_train", "Images_train")
VAL_IMG_DIR = os.path.join(DATA_DIR, "Images_val", "Images_val")
EVAL_CAP_JSON = os.path.join(DATA_DIR, "VRSBench_EVAL_Cap.json")

# --- Output directories (Kaggle persists /kaggle/working/ as notebook output) ---
OUTPUT_DIR = "/kaggle/working/moondream2_vrsbench_final"          # last checkpoint at time cap
BEST_OUTPUT_DIR = "/kaggle/working/moondream2_vrsbench_best"      # best BERT-BLEU4 checkpoint seen
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(BEST_OUTPUT_DIR, exist_ok=True)

In [6]:
# --- Model + tokenizer download (pulled once here, used by every later cell) ---
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

MODEL_NAME = "vikhyatk/moondream2"
MD_REVISION = "2024-08-26"   # pin a revision — moondream2 weights/API have changed across revisions

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MD_REVISION)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    revision=MD_REVISION,
    trust_remote_code=True,
    torch_dtype=DTYPE,
    attn_implementation="eager",   # flash-attn optional; eager is the safe default for finetuning
).to(DEVICE)

# Moondream2 is two sub-modules under one AutoModelForCausalLM: a frozen-by-default vision encoder
# (model.vision_encoder) and a Phi-based text decoder (model.text_model). Only text_model gets LoRA
# here, matching the official finetuning notebook (vision encoder stays frozen).
for p in model.vision_encoder.parameters():
    p.requires_grad = False

print(f"Model + tokenizer downloaded. Elapsed: {elapsed_min():.1f} min")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

...

modeling_phi.py: 0.00B [00:00, ?B/s]

vision_encoder.py: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.74G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model + tokenizer downloaded. Elapsed: 0.7 min


## 2. Build DataFrame from train captions

In [7]:
DATA_DIR = "/kaggle/input/datasets/ayaanmustafa/vrs-bench/vrsbench_data"
ANNOT_FILE = os.path.join(DATA_DIR, "VRSBench_train.json")  # update filename if different
IMG_DIR = os.path.join(DATA_DIR, "Images_train", "Images_train")

with open(ANNOT_FILE, "r") as f:
    raw = json.load(f)

caption_rows = []

for item in raw:
    conversations = item["conversations"]

    if "[caption]" in conversations[0]["value"].lower():
        caption_rows.append({
            "id": item["id"],
            "image": item["image"],
            "caption": conversations[1]["value"]
        })

df = pd.DataFrame(caption_rows)
# Normalize columns: expect a filename column and a caption column
# (adjust the renames below to match the printed column names above)
rename_map = {}
for c in df.columns:
    lc = c.lower()
    if "image" in lc and "path" not in lc:
        rename_map[c] = "filename"
    elif lc in ("image_path", "path"):
        rename_map[c] = "filename"
    elif "caption" in lc:
        rename_map[c] = "caption"

df = df.rename(columns=rename_map)
df["image_path"] = df["filename"].apply(lambda x: os.path.join(IMG_DIR, os.path.basename(str(x))))

df = df[["image_path", "caption"]].dropna().reset_index(drop=True)
print(df.shape)
df.head()

(20264, 2)


,image_path,caption
0,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,"The image, sourced from GoogleEarth, shows a r..."
1,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,This aerial image from Google Earth shows a de...
2,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,"The image, sourced from GoogleEarth, showcases..."
3,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,"The image, sourced from GoogleEarth, shows a f..."
4,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,"The image, sourced from GoogleEarth, showcases..."


## 3. Train set — full dataset

In [8]:
from sklearn.model_selection import train_test_split

train_df, _ = train_test_split(df, test_size=0.1, random_state=SEED)

# Full dataset — no subsampling. Training is wall-clock capped (Section 9), so the full pool
# just means more of it gets seen; the time callback still governs when training stops.
train_df = train_df.reset_index(drop=True)

print("Train:", len(train_df))

Train: 18237


In [9]:
!pip install -q --upgrade "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 35.5 MB/s eta 0:00:00


## 4. LoRA setup
Rank 16 / alpha 16 — same as the baseline run — applied to `text_model` only (vision encoder
frozen, matching the official moondream finetuning notebook). Kept at rank 16 rather than bumped
to 64: the reports' rank-64 gains showed up on the *numeric/grounding* pipelines (SAM3-based,
heavier compute budget per sample), not on plain captioning SFT, and a bigger adapter would eat
into the shorter 8hr budget's step count for no clear captioning benefit. Phi fuses QKV into
`Wqkv` and uses `fc1`/`fc2` for the MLP (different naming from SmolVLM's Llama-style modules).

In [10]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["Wqkv", "out_proj", "fc1", "fc2"],  # Phi-style module names used by moondream2's text_model
    task_type="CAUSAL_LM",
)
model.text_model = get_peft_model(model.text_model, lora_config)
model.text_model.print_trainable_parameters()

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


trainable params: 12,582,912 || all params: 1,430,853,632 || trainable%: 0.8794


## 5. Metric-aware prompt
Median caption length in VRSBench is 53 words — stated explicitly in the prompt, matching both
reference reports' "metric-aware few-shot" framing (GeoNLI Table I: metric-aware prompting lifted
BERT-BLEU4 from 0.43→0.61 and 0.12→0.59 on the two Qwen3-VL rows). Moondream2 doesn't use a chat
template; its convention (and the one used by public finetunes like `gokaygokay/moondream-prompt`)
is a plain `"\n\nQuestion: {q}\n\nAnswer:"` / `" {a}<|endoftext|>"` format around a single
embedded image, with no image few-shot support in the stock API (each call takes exactly one
image). So the few-shot-with-images approach is dropped — the 53-word target instruction goes
directly in the question text instead.

In [11]:
MEDIAN_WORDS = 53
ANSWER_EOS = "<|endoftext|>"

QUESTION_TEXT = (
    "Describe this satellite/aerial image in ONE concise paragraph of about "
    f"{MEDIAN_WORDS} words. Mention key objects, their approximate spatial layout "
    "(e.g. top-left, bottom-right, center), and scene context. Do not hedge, apologize, "
    "or comment on image quality. Do not repeat phrases."
)

## 6. PyTorch Dataset
Same loose safety cap (90 words) as the baseline run — not the primary length-control mechanism,
just a guard against one outlier caption dominating a batch. The length-penalty loss (Section 8,
now at a lower weight) does the actual length-shaping via gradient signal.

In [12]:
from torch.utils.data import Dataset

class VRSBenchMoondreamDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        caption = row["caption"]
        words = caption.split()
        if len(words) > 90:   # loose safety cap only, not the primary length-control mechanism
            caption = " ".join(words[:90])
        return {"image": image, "question": QUESTION_TEXT, "answer": caption}


train_dataset = VRSBenchMoondreamDataset(train_df)
print(len(train_dataset))

18237


## 7. Collator
Follows the official moondream finetuning notebook's sequence layout: `[BOS] + IMG_TOKENS(-100) +
question_tokens(-100) + answer_tokens(labeled) + EOS`. Images go through `model.vision_encoder`'s own
preprocessing, and the 729 image-embedding slots are masked out of the loss (`-100`) since they
aren't token ids.

In [13]:
IMG_TOKENS = 729

def collate_fn(examples):
    # Moondream2 preprocess() accepts ONE PIL image at a time.
    images = torch.stack([
        model.vision_encoder.preprocess(ex["image"])
        for ex in examples
    ]).to(DEVICE, dtype=DTYPE)

    labels_acc, tokens_acc = [], []

    for ex in examples:
        # BOS
        toks = [tokenizer.bos_token_id]

        # BOS + 729 image embedding slots are not language-model targets
        labs = [-100] * (IMG_TOKENS + 1)

        # Question
        q_ids = tokenizer(
            f"\n\nQuestion: {ex['question']}\n\nAnswer:",
            add_special_tokens=False
        ).input_ids

        toks.extend(q_ids)
        labs.extend([-100] * len(q_ids))

        # Answer
        a_ids = tokenizer(
            f" {ex['answer']}{ANSWER_EOS}",
            add_special_tokens=False
        ).input_ids

        toks.extend(a_ids)
        labs.extend(a_ids)

        tokens_acc.append(toks)
        labels_acc.append(labs)

    # Pad batch
    max_len = max(len(x) for x in labels_acc)

    attn_mask_acc = []

    for i in range(len(examples)):
        pad = max_len - len(labels_acc[i])

        labels_acc[i].extend([-100] * pad)
        tokens_acc[i].extend([tokenizer.eos_token_id] * pad)

        attn_mask_acc.append(
            [1] * (max_len - pad) + [0] * pad
        )

    return {
        "images": images,
        "tokens": torch.tensor(
            tokens_acc,
            dtype=torch.long,
            device=DEVICE
        ),
        "labels": torch.tensor(
            labels_acc,
            dtype=torch.long,
            device=DEVICE
        ),
        "attn_mask": torch.tensor(
            attn_mask_acc,
            dtype=torch.bool,
            device=DEVICE
        ),
    }

## 8. Length-penalty loss — penalizes deviation from the median (53 words)

**Lowered from `0.05` → `0.02`.** The 0.05 run was a firmer pull than necessary: at that weight the
auxiliary term competes hard enough with cross-entropy that captions get squeezed toward 53 words
even when the content genuinely needs more or fewer — you lose detail ("dark rooftops", "large
yards") to hit a word count. `0.02` keeps a real gradient pull toward the median (it's not zero) but
lets cross-entropy dominate, so length-shaping is a soft nudge rather than the main optimization
target. Same mechanism as before: an explicit auxiliary term added on top of the model's own
cross-entropy loss, penalizing the squared deviation of each sample's *answer* token count from a
target token count corresponding to `MEDIAN_WORDS`. Since moondream2's `text_model.forward` doesn't
route through a `Trainer.compute_loss` hook, this is computed directly inside the manual training
loop (Section 9) rather than in a `Trainer` subclass.

In [14]:
SAMPLE_TEXT = " ".join(["word"] * MEDIAN_WORDS)
TARGET_TOKEN_COUNT = len(tokenizer(SAMPLE_TEXT, add_special_tokens=False)["input_ids"])
print("Target token count for median length:", TARGET_TOKEN_COUNT)

LENGTH_PENALTY_LAMBDA = 0.025   # lowered from 0.05 — soft pull, not a dominant loss term

def length_penalty(labels, response_lengths):
    deviation = (response_lengths - TARGET_TOKEN_COUNT) ** 2
    return deviation.mean() / (TARGET_TOKEN_COUNT ** 2)   # normalized so it stays a small, stable term


def compute_loss(batch):
    images, tokens, labels, attn_mask = batch["images"], batch["tokens"], batch["labels"], batch["attn_mask"]

    with torch.no_grad():
        img_embs = model.vision_encoder(images)

    tok_embs = model.text_model.get_input_embeddings()(tokens)
    # Splice image embeddings in right after the BOS token, matching moondream2's own encode_image
    # + answer_question conventions (and the official finetuning notebook's collator).
    inputs_embeds = torch.cat((tok_embs[:, 0:1, :], img_embs, tok_embs[:, 1:, :]), dim=1)

    outputs = model.text_model(
        inputs_embeds=inputs_embeds,
        labels=labels,
        attention_mask=attn_mask,
    )
    ce_loss = outputs.loss

    response_lengths = (labels != -100).sum(dim=-1).float()
    len_loss = length_penalty(labels, response_lengths)

    loss = ce_loss + LENGTH_PENALTY_LAMBDA * len_loss
    return loss, ce_loss, len_loss, response_lengths

Target token count for median length: 53


## 9. Metric suite
Moved ahead of training (it used to live in Section 11, after training) so the periodic
best-checkpoint eval in Section 11 can call `compute_all_metrics` / `corpus_bert_bleu4` while
training is still running. **`bert_bleu4` below is unchanged and is the exact formula from both
reference reports**: `BERT-BLEU4 = LP · exp((1/4)Σ log P_n)`, `LP = exp(-0.5·|L_C-L_R|/L_R)` —
geometric mean over n=1..4 of max-cosine BERT n-gram recall, length-penalized. This is the GeoNLI
report's formula exactly (Team 67's report swaps the geometric mean for `max_n P_n`; we match
GeoNLI's since the notebook's `alpha=0.5` / `exp(mean log P_n)` structure already lines up with it
one-to-one).

In [15]:
from sentence_transformers import SentenceTransformer

bert_model = SentenceTransformer("all-MiniLM-L6-v2", device=str(DEVICE))

def get_ngrams(tokens, n):
    return [" ".join(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

def bert_bleu4(candidate, reference, alpha=0.5):
    c_tokens = candidate.lower().split()
    r_tokens = reference.lower().split()
    c_ngrams_by_n = [get_ngrams(c_tokens, n) for n in range(1, 5)]
    r_ngrams_by_n = [get_ngrams(r_tokens, n) for n in range(1, 5)]
    all_texts = [g for grams in c_ngrams_by_n + r_ngrams_by_n for g in grams]
    if not all_texts:
        return 0.0
    embs = bert_model.encode(all_texts, convert_to_tensor=True, show_progress_bar=False, batch_size=256)
    embs = F.normalize(embs, dim=-1)
    idx = 0
    c_embs_by_n, r_embs_by_n = [], []
    for grams in c_ngrams_by_n:
        c_embs_by_n.append(embs[idx: idx + len(grams)]); idx += len(grams)
    for grams in r_ngrams_by_n:
        r_embs_by_n.append(embs[idx: idx + len(grams)]); idx += len(grams)
    log_p_sum = 0.0
    for n in range(4):
        C_emb, R_emb = c_embs_by_n[n], r_embs_by_n[n]
        if len(C_emb) == 0 or len(R_emb) == 0:
            p_n = 1e-9
        else:
            sims = torch.mm(R_emb, C_emb.T)
            p_n = max(sims.max(dim=1).values.mean().item(), 1e-9)
        log_p_sum += np.log(p_n)
    L_c, L_r = len(c_tokens), max(len(r_tokens), 1)
    lp = np.exp(-alpha * abs(L_c - L_r) / L_r)
    return lp * np.exp(log_p_sum / 4)

def corpus_bert_bleu4(candidates, references):
    return float(np.mean([bert_bleu4(c, r) for c, r in zip(candidates, references)]))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [16]:
import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("punkt", quiet=True)

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from pycocoevalcap.cider.cider import Cider

smoothie = SmoothingFunction().method4
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def compute_bleu_n(candidates, references, n):
    weights = tuple([1.0 / n] * n + [0.0] * (4 - n))
    scores = []
    for c, r in zip(candidates, references):
        c_tok, r_tok = c.lower().split(), [r.lower().split()]
        scores.append(sentence_bleu(r_tok, c_tok, weights=weights, smoothing_function=smoothie))
    return float(np.mean(scores))

def compute_meteor(candidates, references):
    return float(np.mean([meteor_score([r.lower().split()], c.lower().split()) for c, r in zip(candidates, references)]))

def compute_rouge_l(candidates, references):
    return float(np.mean([rouge.score(r, c)["rougeL"].fmeasure for c, r in zip(candidates, references)]))

def compute_cider(candidates, references):
    gts = {i: [r] for i, r in enumerate(references)}
    res = {i: [c] for i, c in enumerate(candidates)}
    score, _ = Cider().compute_score(gts, res)
    return float(score)

def compute_avg_len(candidates):
    return float(np.mean([len(c.split()) for c in candidates]))

def compute_all_metrics(candidates, references):
    return {
        "BLEU-1": compute_bleu_n(candidates, references, 1),
        "BLEU-2": compute_bleu_n(candidates, references, 2),
        "BLEU-3": compute_bleu_n(candidates, references, 3),
        "BLEU-4": compute_bleu_n(candidates, references, 4),
        "METEOR": compute_meteor(candidates, references),
        "ROUGE_L": compute_rouge_l(candidates, references),
        "CIDEr": compute_cider(candidates, references),
        "Avg_L": compute_avg_len(candidates),
        "BERT-BLEU4": corpus_bert_bleu4(candidates, references),
    }

## 10. Load official val split + inference helper
Also moved ahead of training (was Sections 12–13 after training) so the periodic eval in
Section 11 has `eval_subset_df` / `generate_caption` available. Uses moondream2's own
`encode_image` + `answer_question` API, so the LoRA adapter is exercised through the model's
native generation path. No hard word-count truncation at inference — the point is to see what the
(now lighter) length-penalty loss learned on its own.

In [17]:
with open(EVAL_CAP_JSON, "r") as f:
    eval_raw = json.load(f)

eval_df = pd.DataFrame(eval_raw)
if "type" in eval_df.columns:
    eval_df = eval_df[eval_df["type"] == "caption"].reset_index(drop=True)

eval_rename = {}
for c in eval_df.columns:
    lc = c.lower()
    if "image" in lc and "path" not in lc:
        eval_rename[c] = "filename"
    elif lc in ("image_path", "path"):
        eval_rename[c] = "filename"
    elif "caption" in lc or lc in ("ground_truth", "answer"):
        eval_rename[c] = "caption"

eval_df = eval_df.rename(columns=eval_rename)
eval_df["image_path"] = eval_df["filename"].apply(lambda x: os.path.join(VAL_IMG_DIR, os.path.basename(str(x))))
eval_df = eval_df[["image_path", "caption"]].dropna().reset_index(drop=True)

EVAL_SUBSET_SIZE = 50          # full official-style eval, used at the very end
QUICK_EVAL_SUBSET_SIZE = 12    # small/fast subset, used repeatedly during training for checkpoint selection

eval_subset_df = eval_df.sample(n=min(EVAL_SUBSET_SIZE, len(eval_df)), random_state=SEED).reset_index(drop=True)
quick_eval_df = eval_subset_df.sample(n=min(QUICK_EVAL_SUBSET_SIZE, len(eval_subset_df)), random_state=SEED).reset_index(drop=True)
print("Full eval samples:", len(eval_subset_df), "| Quick eval samples (during training):", len(quick_eval_df))

Full eval samples: 50 | Quick eval samples (during training): 12


In [18]:
@torch.no_grad()
def generate_caption(image, max_new_tokens=120):
    model.text_model.eval()
    image_embeds = model.encode_image(image)
    answer = model.answer_question(
        image_embeds,
        QUESTION_TEXT,
        tokenizer=tokenizer,
        max_new_tokens=max_new_tokens,
    )
    model.text_model.train()
    return answer.strip()


def validate(eval_df_subset):
    preds, refs = [], []
    for _, row in eval_df_subset.iterrows():
        image = Image.open(row["image_path"]).convert("RGB")
        pred = generate_caption(image)
        preds.append(pred)
        refs.append(row["caption"])
    return preds, refs

## 11. Training — wall-clock capped to **8hr**, best-checkpoint selection

**Lowered from 9hr → 8hr** (`TARGET_TRAIN_MINUTES = 480`). Everything else about the loop —
manual (no `Trainer`), gradient accumulation to effective batch 16, cosine LR schedule with
warmup — is unchanged from the baseline run.

New in this version: every `EVAL_EVERY_MINUTES` (30 min) of wall-clock time, training pauses
briefly, runs the quick 12-sample val eval, computes BERT-BLEU4 with the exact same
`corpus_bert_bleu4` used in the final report-matching eval, and — **only if it's the best score
seen so far** — saves the LoRA adapter to `BEST_OUTPUT_DIR`. This directly targets "best model in
8hrs": with a tighter time budget than the 9hr baseline, there's more risk of the wall-clock cutoff
landing mid-overfit or mid-regression on a bad batch run; checkpointing on an improving metric
guards against handing back a worse model than one seen 90 minutes earlier.

In [19]:
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup
import bitsandbytes as bnb

TARGET_TRAIN_MINUTES = 480   # 8hr wall-clock cap on training (lowered from 9hr)
PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 16        # effective batch still 16, matching the baseline run
LEARNING_RATE = 2e-4
WARMUP_STEPS = 50
NUM_EPOCHS_UPPER_BOUND = 50  # high upper bound — the time cap below stops training first
LOGGING_STEPS = 20
EVAL_EVERY_MINUTES = 30      # periodic quick-eval + best-checkpoint cadence
CHECKPOINT_DIR = "/kaggle/working/moondream2_vrsbench_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

train_loader = DataLoader(
    train_dataset, batch_size=PER_DEVICE_BATCH_SIZE, shuffle=True, collate_fn=collate_fn
)

trainable_params = [p for p in model.text_model.parameters() if p.requires_grad]
optimizer = bnb.optim.AdamW8bit(trainable_params, lr=LEARNING_RATE)

steps_per_epoch = max(1, len(train_loader) // GRAD_ACCUM_STEPS)
total_steps_upper_bound = steps_per_epoch * NUM_EPOCHS_UPPER_BOUND
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=total_steps_upper_bound
)

model.text_model.train()
model.text_model.enable_input_require_grads()   # required: we manually pass inputs_embeds into the checkpointed model
model.text_model.gradient_checkpointing_enable()
if hasattr(model.text_model, "config"):
    model.text_model.config.use_cache = False    # required when using gradient checkpointing

train_start = time.time()
target_seconds = TARGET_TRAIN_MINUTES * 60
next_eval_at_seconds = EVAL_EVERY_MINUTES * 60
global_step = 0
micro_step = 0
stop_training = False
best_bert_bleu4 = -1.0
eval_history = []   # (elapsed_min, BERT-BLEU4) — for a quick sanity plot/printout after the run

for epoch in range(NUM_EPOCHS_UPPER_BOUND):
    if stop_training:
        break
    for batch in train_loader:
        loss, ce_loss, len_loss, response_lengths = compute_loss(batch)
        (loss / GRAD_ACCUM_STEPS).backward()
        micro_step += 1

        if micro_step % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            if global_step % LOGGING_STEPS == 0:
                print(
                    f"step {global_step} | loss {loss.item():.4f} | ce {ce_loss.item():.4f} | "
                    f"len {len_loss.item():.4f} | avg_resp_tok {response_lengths.mean().item():.1f} | "
                    f"elapsed {elapsed_min():.1f} min"
                )

        elapsed_seconds = time.time() - train_start

        # --- periodic quick eval + best-checkpoint save ---
        if elapsed_seconds >= next_eval_at_seconds:
            preds, refs = validate(quick_eval_df)
            quick_score = corpus_bert_bleu4(preds, refs)
            eval_history.append((elapsed_seconds / 60, quick_score))
            print(f"  [eval @ {elapsed_seconds/60:.1f} min] quick BERT-BLEU4 = {quick_score:.4f} "
                  f"(best so far: {max(best_bert_bleu4, quick_score):.4f})")
            if quick_score > best_bert_bleu4:
                best_bert_bleu4 = quick_score
                model.text_model.save_pretrained(BEST_OUTPUT_DIR)
                tokenizer.save_pretrained(BEST_OUTPUT_DIR)
                print(f"  -> new best, saved to {BEST_OUTPUT_DIR}")
            next_eval_at_seconds += EVAL_EVERY_MINUTES * 60

        if elapsed_seconds >= target_seconds:
            print(f"Hit {TARGET_TRAIN_MINUTES}-minute wall-clock cap at step {global_step}, epoch {epoch}.")
            stop_training = True
            break

print(f"Training finished. Total elapsed: {elapsed_min():.1f} min | best quick-eval BERT-BLEU4: {best_bert_bleu4:.4f}")
print("Eval history (elapsed_min, BERT-BLEU4):", eval_history)

step 20 | loss 2.5506 | ce 2.5505 | len 0.0032 | avg_resp_tok 56.0 | elapsed 23.2 min
  [eval @ 30.0 min] quick BERT-BLEU4 = 0.4224 (best so far: 0.4224)
  -> new best, saved to /kaggle/working/moondream2_vrsbench_best


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:372: UserWarning: Could not find a config file in  - will assume that the vocabulary was not modified.
  warnings.warn(


step 40 | loss 1.8497 | ce 1.8454 | len 0.1723 | avg_resp_tok 75.0 | elapsed 45.4 min
  [eval @ 60.0 min] quick BERT-BLEU4 = 0.4936 (best so far: 0.4936)
  -> new best, saved to /kaggle/working/moondream2_vrsbench_best
step 60 | loss 1.4795 | ce 1.4680 | len 0.4614 | avg_resp_tok 89.0 | elapsed 67.8 min
step 80 | loss 1.3148 | ce 1.3140 | len 0.0356 | avg_resp_tok 43.0 | elapsed 89.5 min
  [eval @ 90.1 min] quick BERT-BLEU4 = 0.5258 (best so far: 0.5258)
  -> new best, saved to /kaggle/working/moondream2_vrsbench_best
step 100 | loss 1.5465 | ce 1.5330 | len 0.5415 | avg_resp_tok 92.0 | elapsed 111.9 min
  [eval @ 120.0 min] quick BERT-BLEU4 = 0.5526 (best so far: 0.5526)
  -> new best, saved to /kaggle/working/moondream2_vrsbench_best
step 120 | loss 1.6498 | ce 1.6493 | len 0.0174 | avg_resp_tok 60.0 | elapsed 134.3 min
  [eval @ 150.0 min] quick BERT-BLEU4 = 0.6141 (best so far: 0.6141)
  -> new best, saved to /kaggle/working/moondream2_vrsbench_best
step 140 | loss 1.4225 | ce 1.41

## 12. Save final (last-checkpoint) model to the output directory

In [20]:
model.text_model.save_pretrained(OUTPUT_DIR)   # LoRA adapter weights — state at the 8hr cutoff
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved last-checkpoint LoRA adapter + tokenizer to:", OUTPUT_DIR)
print("Best quick-eval checkpoint (if any improvement was seen) is saved separately at:", BEST_OUTPUT_DIR)
!ls {OUTPUT_DIR}
!ls {BEST_OUTPUT_DIR}

Saved last-checkpoint LoRA adapter + tokenizer to: /kaggle/working/moondream2_vrsbench_final
Best quick-eval checkpoint (if any improvement was seen) is saved separately at: /kaggle/working/moondream2_vrsbench_best
adapter_config.json	   merges.txt		    tokenizer_config.json
adapter_model.safetensors  README.md		    tokenizer.json
added_tokens.json	   special_tokens_map.json  vocab.json


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


adapter_config.json	   merges.txt		    tokenizer_config.json
adapter_model.safetensors  README.md		    tokenizer.json
added_tokens.json	   special_tokens_map.json  vocab.json


## 13. Final 50-sample official eval — last checkpoint vs. best checkpoint

Runs the full report-matching eval (same `eval_subset_df`, same `compute_all_metrics` /
`corpus_bert_bleu4` as the baseline notebook) against **both** candidates, and picks whichever
scores higher on BERT-BLEU4 as the model to report as "the final model" — this is the actual
best-in-8hrs selection, not just whatever the clock cut off mid-step.

In [21]:
# --- Eval the LAST checkpoint (already loaded in memory as `model.text_model`) ---
preds_last, refs_last = validate(eval_subset_df)
metrics_last = compute_all_metrics(preds_last, refs_last)
print("=== LAST checkpoint (8hr cutoff) ===")
print(metrics_last)
print(f"Avg predicted length: {np.mean([len(p.split()) for p in preds_last]):.1f} words (target median: {MEDIAN_WORDS})")

# --- Eval the BEST checkpoint (reload adapter weights if a better one was ever saved) ---
if best_bert_bleu4 > -1.0:
    from peft import PeftModel
    # Re-load a clean base text_model, then attach the saved best adapter.
    base_text_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, revision=MD_REVISION, trust_remote_code=True, torch_dtype=DTYPE,
    ).text_model.to(DEVICE)
    model.text_model = PeftModel.from_pretrained(base_text_model, BEST_OUTPUT_DIR).to(DEVICE)
    model.text_model.eval()

    preds_best, refs_best = validate(eval_subset_df)
    metrics_best = compute_all_metrics(preds_best, refs_best)
    print("\n=== BEST checkpoint (periodic eval, highest quick BERT-BLEU4 during training) ===")
    print(metrics_best)
    print(f"Avg predicted length: {np.mean([len(p.split()) for p in preds_best]):.1f} words (target median: {MEDIAN_WORDS})")

    FINAL_MODEL_DIR = "/kaggle/working/moondream2_vrsbench_FINAL_BEST"
    if metrics_best["BERT-BLEU4"] >= metrics_last["BERT-BLEU4"]:
        print(f"\n>>> BEST checkpoint wins ({metrics_best['BERT-BLEU4']:.4f} vs {metrics_last['BERT-BLEU4']:.4f}). "
              f"Copying it to {FINAL_MODEL_DIR} as the final model.")
        import shutil
        shutil.copytree(BEST_OUTPUT_DIR, FINAL_MODEL_DIR, dirs_exist_ok=True)
        final_metrics, final_preds, final_refs = metrics_best, preds_best, refs_best
    else:
        print(f"\n>>> LAST checkpoint wins ({metrics_last['BERT-BLEU4']:.4f} vs {metrics_best['BERT-BLEU4']:.4f}). "
              f"Copying it to {FINAL_MODEL_DIR} as the final model.")
        import shutil
        shutil.copytree(OUTPUT_DIR, FINAL_MODEL_DIR, dirs_exist_ok=True)
        final_metrics, final_preds, final_refs = metrics_last, preds_last, refs_last
else:
    print("No periodic-eval improvement was ever recorded (quick eval never ran or never beat -1.0) "
          "— the last checkpoint is the only candidate.")
    FINAL_MODEL_DIR = OUTPUT_DIR
    final_metrics, final_preds, final_refs = metrics_last, preds_last, refs_last

print(f"\nFINAL MODEL DIR: {FINAL_MODEL_DIR}")
print(f"FINAL BERT-BLEU4 (report-comparable): {final_metrics['BERT-BLEU4']:.4f}")
print(f"Total run time: {elapsed_min():.1f} min")

=== LAST checkpoint (8hr cutoff) ===
{'BLEU-1': 0.34011169185093726, 'BLEU-2': 0.206355643940666, 'BLEU-3': 0.1303714629365245, 'BLEU-4': 0.08933849749462688, 'METEOR': 0.29917975950074355, 'ROUGE_L': 0.3638359298472348, 'CIDEr': 0.2472845915693111, 'Avg_L': 42.84, 'BERT-BLEU4': 0.5704011844475351}
Avg predicted length: 42.8 words (target median: 53)

=== BEST checkpoint (periodic eval, highest quick BERT-BLEU4 during training) ===
{'BLEU-1': 0.3510794647814338, 'BLEU-2': 0.2055952104925747, 'BLEU-3': 0.12401756597777495, 'BLEU-4': 0.0792269863411213, 'METEOR': 0.28636015283681115, 'ROUGE_L': 0.3654220463367109, 'CIDEr': 0.3128429705913296, 'Avg_L': 41.02, 'BERT-BLEU4': 0.5864846533734662}
Avg predicted length: 41.0 words (target median: 53)

>>> BEST checkpoint wins (0.5865 vs 0.5704). Copying it to /kaggle/working/moondream2_vrsbench_FINAL_BEST as the final model.

FINAL MODEL DIR: /kaggle/working/moondream2_vrsbench_FINAL_BEST
FINAL BERT-BLEU4 (report-comparable): 0.5865
Total run t

## 14. Score comparison vs. the reference reports

| Source | Model / setting | BERT-BLEU4 |
|---|---|---|
| GeoNLI report | Qwen3-VL-4B, standard few-shot | 0.4333 |
| GeoNLI report | Qwen3-VL-4B, metric-aware few-shot | 0.6120 |
| GeoNLI report | Qwen2.5-VL-7B-4bit, SFT + metric-aware (best) | **0.6841** |
| Team 67 report | Qwen3-VL (base) | 0.7522 |
| Team 67 report | Ours — unified end-to-end, fine-tuned (best) | **0.7993** |
| This notebook | Moondream2 + LoRA, 8hr, λ=0.02 (this run) | *printed above as `final_metrics['BERT-BLEU4']`* |

Run the cell above and drop the printed `final_metrics['BERT-BLEU4']` into the table to see where
this run lands relative to both reports. Note Moondream2 (≈2B, frozen vision encoder, LoRA-only
text-model tuning) is a much smaller/cheaper setup than either report's winning configuration
(7B-class SFT models) — closing the full gap to 0.7993 in 8hr on a single smaller model is
unlikely, but the periodic best-checkpoint selection plus the lighter length penalty should move
this run closer to the GeoNLI SFT number (0.6841) than the original 0.05-penalty / 9hr run did.

## 15. Sample predictions

In [22]:
for p, r in list(zip(final_preds, final_refs))[:5]:
    print(f"PRED ({len(p.split())}w):", p)
    print(f"REF  ({len(r.split())}w):", r)
    print("-" * 60)

PRED (36w): The image, sourced from GoogleEarth, features a large ship docked at a harbor. The ship is positioned in the middle-right area of the image, with a distinct structure extending from the right side of the harbor.
REF  (24w): The image from GoogleEarth shows a docking area with a ship. The ship has a white and green color scheme and is very long.
------------------------------------------------------------
PRED (32w): The image, sourced from GoogleEarth, features a small basketball court located in the bottom-right area of the frame. The court is surrounded by a grassy area and is marked with white lines.
REF  (35w): This aerial image from GoogleEarth features a variety of elements including a prominent basketball court. The basketball court is prominently placed in the lower central part of the image, paved in gray with white markings.
------------------------------------------------------------
PRED (51w): The image, sourced from GoogleEarth, features a small harbor with se